# EdgeCompose-VLM - exploratory analysis

Loads the per-query logs and aggregate table produced by `scripts/benchmark.py` / `scripts/analyze.py`. All heavy lifting lives in the `edgecompose` package; this notebook is only for interactive inspection.

In [ ]:
import os, sys
os.environ.setdefault('MKL_THREADING_LAYER', 'SEQUENTIAL')  # see edgecompose/__init__.py
sys.path.insert(0, '..')
import pandas as pd
from edgecompose.analysis.aggregate import load_raw, aggregate
from pathlib import Path
N = 1000  # manifest size (200 = dev subsets)
df = load_raw(Path('../results/raw'), N)
agg = aggregate(df)
agg[['dataset','config_name','quality','ttft_ms_p50','total_latency_ms_p50','peak_allocated_mb_p50']]

## Stage-wise medians per configuration

In [ ]:
cols = ['preprocess_ms_p50','vision_ms_p50','compress_ms_p50','prefill_ms_p50','decode_ms_p50','total_latency_ms_p50']
agg.set_index(['dataset','config_name'])[cols].round(1)

## Where does compression change the answer?
Paired comparison of TextVQA answers between the baseline and 25% VisionZip.

In [ ]:
t = df[(df.dataset=='textvqa') & df.config_name.isin(['C0_sdpa_r100','C3_sdpa_r25'])]
p = t.pivot(index='sample_id', columns='config_name', values='score')
print('improved:', (p.C3_sdpa_r25 > p.C0_sdpa_r100).sum(), ' worsened:', (p.C3_sdpa_r25 < p.C0_sdpa_r100).sum())
q = t.pivot(index='sample_id', columns='config_name', values='prediction')
q[p.C3_sdpa_r25 < p.C0_sdpa_r100].head(15)

## Prefill latency vs prefill length (AWQ kernel dispatch)
AutoAWQ switches from the Triton split-K GEMM to dequantize + cuBLAS when `batch*seq >= 1024`.

In [ ]:
import matplotlib.pyplot as plt
ok = df[(df.status=='ok') & (df.attention_backend=='sdpa')]
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(ok.prefill_seq_len, ok.prefill_ms, s=6, alpha=.4)
ax.axvline(1024, color='gray', ls='--'); ax.set_xlabel('prefill tokens'); ax.set_ylabel('prefill ms'); ax.set_ylim(bottom=0)